In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact
import nibabel as nib

import pydicom
import scipy.ndimage
# import gdcm

import glob

from skimage import measure 
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from skimage.morphology import disk, opening, closing
from tqdm import tqdm

from IPython.display import HTML
from PIL import Image

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import os
from pathlib import Path
import pydicom

In [3]:
# Function to load and sort DICOM slices robustly
def load_scans(dcm_path):
    # List all .dcm files in the directory
    files = [os.path.join(dcm_path, f) for f in os.listdir(dcm_path) if f.lower().endswith('.dcm')]
    
    # Read only DICOMs that contain pixel data (skip reports, RTSTRUCT, etc.)
    slices = []
    for f in files:
        ds = pydicom.dcmread(f)
        if hasattr(ds, "PixelData"):
            slices.append(ds)

    # Define a safe sort key
    def sort_key(ds):
        if hasattr(ds, "ImagePositionPatient") and ds.ImagePositionPatient is not None:
            return float(ds.ImagePositionPatient[2])
        elif hasattr(ds, "SliceLocation"):
            return float(ds.SliceLocation)
        elif hasattr(ds, "InstanceNumber"):
            return int(ds.InstanceNumber)
        else:
            return 0  # fallback if nothing is available

    # Sort slices
    slices.sort(key=sort_key)

    return slices

root = Path(r"./Dataset/train")

# For testing: set to a patient ID or None for all
test_patient = None  # e.g., "ID0001"

all_series = []

for patient in sorted(os.listdir(root)):
    if test_patient and patient != test_patient:
        continue
    patient_path = root / patient
    if not patient_path.is_dir():
        continue


    # Collect all DICOM files in this series
    dicoms = [f for f in patient_path.iterdir() if f.suffix.lower() == ".dcm"]
    if dicoms:
        all_series.append({
            "patient_id": patient,
            "patient_path": patient_path,
            "dicom_files": dicoms
        })

print(f"Found {len(all_series)} series")
print("Example:", all_series[0]["patient_path"] if all_series else "No series found")
series_path = all_series[0]["patient_path"]
scans = load_scans(patient_path)
print(f"Loaded {len(scans)} DICOM slices.")
print(scans[2])  # Print the first slice's metadata for verification

Found 8 series
Example: Dataset/train/ID00336637202286801879145
Loaded 69 DICOM slices.
Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 206
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: CT Image Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.2.276.0.7230010.3.1.4.0.37492.1591342969.177417
(0002,0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002,0012) Implementation Class UID            UI: 1.2.276.0.7230010.3.0.3.6.1
(0002,0013) Implementation Version Name         SH: 'OSIRIX_361'
(0002,0016) Source Application Entity Title     AE: 'ANONYMOUS'
-------------------------------------------------
(0008,0005) Specific Character Set              CS: 'ISO_IR 100'
(0008,0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY', 'AXIAL']
(0008,0018) SOP Instance UID                    UI: 1.2.276.0.7230010.3.1.4.0.37492.

In [7]:
all_series[0]["dicom_files"]

[PosixPath('Dataset/train/ID00336637202286801879145/2.dcm'),
 PosixPath('Dataset/train/ID00336637202286801879145/29.dcm'),
 PosixPath('Dataset/train/ID00336637202286801879145/16.dcm'),
 PosixPath('Dataset/train/ID00336637202286801879145/21.dcm'),
 PosixPath('Dataset/train/ID00336637202286801879145/25.dcm'),
 PosixPath('Dataset/train/ID00336637202286801879145/27.dcm'),
 PosixPath('Dataset/train/ID00336637202286801879145/10.dcm'),
 PosixPath('Dataset/train/ID00336637202286801879145/17.dcm'),
 PosixPath('Dataset/train/ID00336637202286801879145/18.dcm'),
 PosixPath('Dataset/train/ID00336637202286801879145/5.dcm'),
 PosixPath('Dataset/train/ID00336637202286801879145/9.dcm'),
 PosixPath('Dataset/train/ID00336637202286801879145/13.dcm'),
 PosixPath('Dataset/train/ID00336637202286801879145/33.dcm'),
 PosixPath('Dataset/train/ID00336637202286801879145/19.dcm'),
 PosixPath('Dataset/train/ID00336637202286801879145/11.dcm'),
 PosixPath('Dataset/train/ID00336637202286801879145/24.dcm'),
 PosixPath(

In [4]:
# Convert pixel arrays to Hounsfield Units (HU)

def set_outside_scanner_to_air(raw_pixelarrays):
    raw_pixelarrays[raw_pixelarrays <= -1500] = 0
    return raw_pixelarrays

def transform_to_hu(slices):
    images = np.stack([file.pixel_array for file in slices])
    images = images.astype(np.int16)
    print(images)

    #images = set_outside_scanner_to_air(images)
    
    # convert to HU
    for n in range(len(slices)):
        
        intercept = slices[n].RescaleIntercept
        slope = slices[n].RescaleSlope
        
        if slope != 1:
            images[n] = slope * images[n].astype(np.float64)
            images[n] = images[n].astype(np.int16)
            
        images[n] += np.int16(intercept)
     #Clip    
    # images = np.clip(images, a_min = -1000,a_max = 1000)
    return np.array(images, dtype=np.int16)

# Usage example:
hu_scans = transform_to_hu(scans)
print(hu_scans)  # (num_slices, height, width)
print(np.min(hu_scans), np.max(hu_scans))
from ctviewer import CTViewer
  # assuming that volumetric_image is the 3-dimensional numpy array
CTViewer(hu_scans)

[[[-3024 -3024 -3024 ... -3024 -3024 -3024]
  [-3024 -3024 -3024 ... -3024 -3024 -3024]
  [-3024 -3024 -3024 ... -3024 -3024 -3024]
  ...
  [-3024 -3024 -3024 ... -3024 -3024 -3024]
  [-3024 -3024 -3024 ... -3024 -3024 -3024]
  [-3024 -3024 -3024 ... -3024 -3024 -3024]]

 [[-3024 -3024 -3024 ... -3024 -3024 -3024]
  [-3024 -3024 -3024 ... -3024 -3024 -3024]
  [-3024 -3024 -3024 ... -3024 -3024 -3024]
  ...
  [-3024 -3024 -3024 ... -3024 -3024 -3024]
  [-3024 -3024 -3024 ... -3024 -3024 -3024]
  [-3024 -3024 -3024 ... -3024 -3024 -3024]]

 [[-3024 -3024 -3024 ... -3024 -3024 -3024]
  [-3024 -3024 -3024 ... -3024 -3024 -3024]
  [-3024 -3024 -3024 ... -3024 -3024 -3024]
  ...
  [-3024 -3024 -3024 ... -3024 -3024 -3024]
  [-3024 -3024 -3024 ... -3024 -3024 -3024]
  [-3024 -3024 -3024 ... -3024 -3024 -3024]]

 ...

 [[-3024 -3024 -3024 ... -3024 -3024 -3024]
  [-3024 -3024 -3024 ... -3024 -3024 -3024]
  [-3024 -3024 -3024 ... -3024 -3024 -3024]
  ...
  [-3024 -3024 -3024 ... -3024 -3024 -30

interactive(children=(RadioButtons(description='Slice plane selection:', index=2, options=('y-z', 'z-x', 'x-y'…

In [5]:
def get_original_spacing(scans):

    slice_thickness = float(scans[0].SliceThickness)
    pixel_spacing = [float(sp) for sp in scans[0].PixelSpacing]  # [row_spacing, col_spacing]

    # Return as (z, y, x)
    return (slice_thickness, pixel_spacing[0], pixel_spacing[1])

#slice_thickness, spacing_y, spacing_x =  get_original_spacing(scans)
slice_thickness, spacing_y, spacing_x =  get_original_spacing(scans)
affine = np.diag([spacing_x, spacing_y, slice_thickness, 1])

nifti_img = nib.Nifti1Image(hu_scans.astype(np.int16), affine)

In [6]:
nifti_spacing = nifti_img.header.get_zooms()

print("NIfTI spacing:", nifti_spacing)

# Check intensity range
data = nifti_img.get_fdata()
print("HU range:", np.min(data), "to", np.max(data))

NIfTI spacing: (np.float32(0.394531), np.float32(0.394531), np.float32(1.25))
HU range: -3846.0 to 2319.0


# Convert our dicom images to nifti 

1. We will convert dicom series into nifti file. Read more about nifti formats: https://nifti.nimh.nih.gov/background/

*Why Nifti?*
-

In [12]:
# convert the dicom files into nifty format
import os
from pathlib import Path
import numpy as np
import pydicom
import nibabel as nib

# -----------------------------
# Load and sort DICOM slices
# -----------------------------
def load_scans(dcm_path):
    files = [os.path.join(dcm_path, f) for f in os.listdir(dcm_path) if f.endswith('.dcm')]
    slices = [pydicom.dcmread(f) for f in files]
    # Sort by z-coordinate
    slices.sort(key=lambda x: float(x.ImagePositionPatient[2]))
    return slices

# -----------------------------
# Convert pixel arrays to HU
# -----------------------------
def transform_to_hu(slices):
    images = np.stack([file.pixel_array for file in slices])
    images = images.astype(np.int16)

    for n in range(len(slices)):
        intercept = slices[n].RescaleIntercept
        slope = slices[n].RescaleSlope

        if slope != 1:
            images[n] = slope * images[n].astype(np.float64)
            images[n] = images[n].astype(np.int16)

        images[n] += np.int16(intercept)
    #Clip    
    images = np.clip(images, a_min = -1000,a_max = 1000)
    return np.array(images, dtype=np.int16)

# -----------------------------
# Extract voxel spacing
# -----------------------------
def get_original_spacing(scans):
    slice_thickness = float(scans[0].SliceThickness)
    pixel_spacing = [float(sp) for sp in scans[0].PixelSpacing]  # [row_spacing, col_spacing]
    # Return as (z, y, x)
    return (slice_thickness, pixel_spacing[0], pixel_spacing[1])

# -----------------------------
# Main pipeline
# -----------------------------
def convert_all_dicoms_to_nifti(root_dir, test_patient=None):
    root = Path(root_dir)
    all_series = []

    # Collect all series
    for patient in sorted(os.listdir(root)):
        if test_patient and patient != test_patient:
            continue
        patient_path = root / patient
        if not patient_path.is_dir():
            continue

        # Collect all DICOM files in this series
        dicoms = [f for f in patient_path.iterdir() if f.suffix.lower() == ".dcm"]
        if dicoms:
            all_series.append({
                "patient_id": patient,
                "patient_path": patient_path,
                "dicom_files": dicoms
            })

    print(f"Found {len(all_series)} patients with DICOM series.")

    # Process each series
    for series in all_series:
        patient_id = series["patient_id"]
        patient_path = series["patient_path"]
        series_path = series["dicom_files"]
        

        print(f"\nProcessing Patient {patient_id}")

        # Load and convert to HU
        scans = load_scans(patient_path)
        hu_scans = transform_to_hu(scans)

        # Get spacing and affine
        slice_thickness, spacing_y, spacing_x = get_original_spacing(scans)
        affine = np.diag([spacing_x, spacing_y, slice_thickness, 1])

        # Create output folder
        output_folder = "./Dataset/Nifti/"
        os.makedirs(output_folder, exist_ok=True)

        # Save NIfTI
        output_path = os.path.join(output_folder, f"{patient_id}.nii.gz")
        nifti_img = nib.Nifti1Image(hu_scans.astype(np.int16), affine)
        nib.save(nifti_img, str(output_path))

        print(f"Saved NIfTI: {output_path}")
   

# -----------------------------
# Run the pipeline
# -----------------------------
if __name__ == "__main__":
    convert_all_dicoms_to_nifti("./Dataset/train/", test_patient=None)

Found 8 patients with DICOM series.

Processing Patient ID00336637202286801879145
Saved NIfTI: ./Dataset/Nifti/ID00336637202286801879145.nii.gz

Processing Patient ID00337637202286839091062
Saved NIfTI: ./Dataset/Nifti/ID00337637202286839091062.nii.gz

Processing Patient ID00339637202287377736231
Saved NIfTI: ./Dataset/Nifti/ID00339637202287377736231.nii.gz

Processing Patient ID00340637202287399835821
Saved NIfTI: ./Dataset/Nifti/ID00340637202287399835821.nii.gz

Processing Patient ID00341637202287410878488
Saved NIfTI: ./Dataset/Nifti/ID00341637202287410878488.nii.gz

Processing Patient ID00342637202287526592911
Saved NIfTI: ./Dataset/Nifti/ID00342637202287526592911.nii.gz

Processing Patient ID00343637202287577133798
Saved NIfTI: ./Dataset/Nifti/ID00343637202287577133798.nii.gz

Processing Patient ID00344637202287684217717
Saved NIfTI: ./Dataset/Nifti/ID00344637202287684217717.nii.gz


In [ ]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

# -----------------------------------------
# Load your NIfTI CT scan
# -----------------------------------------
nifti_path = "./Dataset/Nifti/ID00344637202287684217717.nii.gz"

nii = nib.load(nifti_path)
ct = nii.get_fdata().astype(np.int16)

print("CT shape (Z, Y, X):", ct.shape)
print("Spacing:", nii.header.get_zooms())

# -----------------------------------------
# AXIAL VIEW VISUALIZATION FUNCTION
# -----------------------------------------
def show_axial(slice_idx):
    plt.figure(figsize=(3,6))
    plt.imshow(ct[slice_idx, :, :], cmap="gray")
    plt.title(f"AXIAL VIEW — Slice {slice_idx}")
    plt.axis("off")
    plt.show()

# -----------------------------------------
# Interactive slider to scroll slices
# -----------------------------------------
interact(
    show_axial,
    slice_idx=widgets.IntSlider(
        min=0,
        max=ct.shape[0]-1,
        step=1,
        value=ct.shape[0]//2
    )
)

CT shape (Z, Y, X): (69, 768, 768)
Spacing: (np.float32(0.394531), np.float32(0.394531), np.float32(1.25))


interactive(children=(IntSlider(value=34, description='slice_idx', max=68), Output()), _dom_classes=('widget-i…

<function __main__.show_axial(slice_idx)>

Now we have our Nifti files

- Let's create lung masks for each patient

In [16]:
import os
from pathlib import Path
import nibabel as nib
import SimpleITK as sitk
from lungmask import LMInferer


# Load NIfTI
nii = nib.load(str(nifti_path))
ct_array = nii.get_fdata().astype("int16")

# Convert to SimpleITK image
sitk_ct = sitk.GetImageFromArray(ct_array)

# NIfTI spacing is (x, y, z). SimpleITK expects (z, y, x)
nifti_spacing = nii.header.get_zooms()[:3]
sitk_ct.SetSpacing(tuple(map(float, (nifti_spacing[2], nifti_spacing[1], nifti_spacing[0]))))

# Run lungmask
segmentation = inferer.apply(sitk_ct)

# Convert segmentation back to NIfTI (whole lung mask)
mask_img = nib.Nifti1Image((segmentation).astype("int16"), nii.affine)

# Create output folder
patient_id = patient
output_folder = './Dataset/masks/'
os.makedirs(output_folder, exist_ok=True)

# Save mask
output_path = os.path.join(output_folder, f"{patient_id}_mask.nii.gz")
nib.save(mask_img, output_path)

4it [02:00, 30.00s/it]                          

lungmask 2026-06-29 10:32:27 Postprocessing



100%|██████████| 2/2 [00:00<00:00, 165.66it/s]


In [ ]:



# -----------------------------
# Initialize lungmask inferer
# -----------------------------
# inferer = LMInferer(modelname="R231CovidWeb")

inferer = LMInferer() # default model is U-net(R231)

# -----------------------------
# Process all patients
# -----------------------------
def generate_masks_from_nifti(root_dir, test_patient=None):
    root = Path(root_dir)
    # p = ['1001138']
    for patient in os.listdir(root_dir):
        if test_patient and patient != test_patient:
            continue

        patient_path = root / patient
        if not patient_path.is_dir():
            continue

        print(f"\nProcessing Patient {patient}")
        
        # Load NIfTI
        nii = nib.load(str(nifti_path))
        ct_array = nii.get_fdata().astype("int16")

        # Convert to SimpleITK image
        sitk_ct = sitk.GetImageFromArray(ct_array)

        # NIfTI spacing is (x, y, z). SimpleITK expects (z, y, x)
        nifti_spacing = nii.header.get_zooms()[:3]
        sitk_ct.SetSpacing(tuple(map(float, (nifti_spacing[2], nifti_spacing[1], nifti_spacing[0]))))

        # Run lungmask
        segmentation = inferer.apply(sitk_ct)

        # Convert segmentation back to NIfTI (whole lung mask)
        mask_img = nib.Nifti1Image((segmentation).astype("int16"), nii.affine)

        # Create output folder
        patient_id = patient
        output_folder = './Dataset/masks/'
        os.makedirs(output_folder, exist_ok=True)

        # Save mask
        output_path = os.path.join(output_folder, f"{patient_id}_mask.nii.gz")
        nib.save(mask_img, output_path)

        print(f"✅ Saved mask: {output_path}")

# -----------------------------
# Run
# -----------------------------
if __name__ == "__main__":
    generate_masks_from_nifti("./Dataset/Nifti/", test_patient=None)

lungmask 2026-06-29 10:27:52 No GPU found, using CPU instead


Use GPUs

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print(torch.__version__)